# 19 Frozen Visual Encoder: VideoMAE / DINO

Purpose: run Colab-only frozen visual embeddings and train a lightweight supervised event head.

Required inputs:

- `manifests/bbe_events_v1.parquet`
- `clips/{FULL_RUN_ID}/clips_v1.parquet` with real `clip_path`

Created outputs:

- `features/video_embedding_v1/manifest.parquet` for VideoMAE, or `features/image_embedding_v1/manifest.parquet` for DINO
- `predictions/video_frozen_encoder_mlb_2024_2026_v2/predictions_v1.parquet`
- `predictions/video_frozen_encoder_mlb_2024_2026_v2/metrics_v1.json`

Model downloads are disabled by default. Use L4 for a pilot. Use A100 for larger encoders, longer clips, or larger batches.


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
PREDICTION_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id', run_id(RUN_PROFILE, 'video_run_id'))
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
IMAGE_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'image_embedding_feature_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('PREDICTION_RUN_ID =', PREDICTION_RUN_ID)
print('VIDEO_EMBEDDING_FEATURE_ID =', VIDEO_EMBEDDING_FEATURE_ID)


In [ ]:
import importlib.util
import subprocess

VISUAL_SETTINGS = stage_settings(RUN_PROFILE, 'frozen_visual_encoder')
INSTALL_TRANSFORMERS = bool(VISUAL_SETTINGS.get('install_transformers_default', True))

if INSTALL_TRANSFORMERS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'pillow', 'opencv-python-headless'])

print('torch available =', importlib.util.find_spec('torch') is not None)
print('transformers available =', importlib.util.find_spec('transformers') is not None)
print('cv2 available =', importlib.util.find_spec('cv2') is not None)


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.runtime.device import summarize_runtime_device

print(summarize_runtime_device(prefer_gpu=True, require_gpu=False))
required = [
    'manifests/bbe_events_v1.parquet',
    f'clips/{FULL_RUN_ID}/clips_v1.parquet',
]
print(check_artifacts(BASE_DIR, required))


In [ ]:
from sport_pipeline.models.video.frozen_embeddings import run_frozen_visual_encoder

VISUAL_SETTINGS = stage_settings(RUN_PROFILE, 'frozen_visual_encoder')
ENCODER = str(VISUAL_SETTINGS.get('encoder', 'videomae'))  # 'dinov3' for contact-frame image embeddings
MODEL_ID = VISUAL_SETTINGS.get('model_id')
ALLOW_MODEL_DOWNLOAD = bool(VISUAL_SETTINGS.get('allow_model_download', True))
MAX_CLIPS = VISUAL_SETTINGS.get('max_clips')
HEAD_EPOCHS = int(VISUAL_SETTINGS.get('head_epochs', 20))
TRUST_REMOTE_CODE = bool(VISUAL_SETTINGS.get('trust_remote_code', False))
NUM_FRAMES = int(VISUAL_SETTINGS.get('num_frames', 16))
IMAGE_SIZE = VISUAL_SETTINGS.get('image_size', 224)
RESUME_VISUAL = bool(VISUAL_SETTINGS.get('resume', True))
CHECKPOINT_EVERY_CLIPS = int(VISUAL_SETTINGS.get('checkpoint_every_clips', 1))
CACHE_INPUTS = bool(VISUAL_SETTINGS.get('cache_inputs', False))
CACHE_NUM_WORKERS = int(VISUAL_SETTINGS.get('cache_num_workers', 4))
CACHE_MIN_FREE_DISK_GB = float(VISUAL_SETTINGS.get('cache_min_free_disk_gb', 20))
CACHE_MAX_FILE_MB = VISUAL_SETTINGS.get('cache_max_file_mb')
FEATURE_DIR_ID = IMAGE_EMBEDDING_FEATURE_ID if ENCODER == 'dinov3' else VIDEO_EMBEDDING_FEATURE_ID

print('encoder =', ENCODER, 'model_id =', MODEL_ID or '(default)', 'trust_remote_code =', TRUST_REMOTE_CODE)
print('frozen_visual cache_inputs =', CACHE_INPUTS, 'cache_num_workers =', CACHE_NUM_WORKERS)

outputs = run_frozen_visual_encoder(
    BASE_DIR,
    clip_run_id=FULL_RUN_ID,
    prediction_run_id=PREDICTION_RUN_ID,
    encoder=ENCODER,
    model_id=MODEL_ID,
    allow_model_download=ALLOW_MODEL_DOWNLOAD,
    max_clips=MAX_CLIPS,
    num_frames=NUM_FRAMES,
    image_size=IMAGE_SIZE,
    head_epochs=HEAD_EPOCHS,
    device='auto',
    trust_remote_code=TRUST_REMOTE_CODE,
    require_non_empty=True,
    resume=RESUME_VISUAL,
    checkpoint_every_clips=CHECKPOINT_EVERY_CLIPS,
    feature_dir_id=FEATURE_DIR_ID,
    cache_dir=CACHE_DIR,
    cache_inputs=CACHE_INPUTS,
    cache_num_workers=CACHE_NUM_WORKERS,
    cache_min_free_disk_gb=CACHE_MIN_FREE_DISK_GB,
    cache_max_file_mb=CACHE_MAX_FILE_MB,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
expected = [
    f'predictions/{PREDICTION_RUN_ID}/predictions_v1.parquet',
    f'predictions/{PREDICTION_RUN_ID}/metrics_v1.json',
    f'reports/preflight/frozen_visual_encoder_{PREDICTION_RUN_ID}_progress.json',
    f'models/video/{PREDICTION_RUN_ID}/linear_head_checkpoint.pt',
]
print(check_artifacts(BASE_DIR, expected))
print('NEXT: notebooks/15_full_fusion.ipynb, with SOURCE_RUNS including sequence_tcn_mlb_2024_2026_v2 and video_frozen_encoder_mlb_2024_2026_v2')
